# Download Data

In [3]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.decomposition import PCA


In [4]:
data = pd.read_csv("fra_perfumes.csv")


# Pre-Cleaning

## Text Cleaning

In [5]:
def bayesian_avg(R, v, C, m):
    """ R = perfume’s average rating
    v = number of votes (rating count)
    C = mean rating across all perfumes
    m = treshold, minimum vote count"""
    return (v/(v+m))*R + (m/(v+m))*C

def clean_text(text):
    return text.replace("[", "").replace("]", "").replace("'", "")

def clean_accords(a):
    if isinstance(a, str):
        return a.replace(",", " ").replace("/", " ")
    return

In [6]:
def clean_data(df):
    data = df.copy()
    
    ## Feature engineering

    data["Name"] = [b.replace(a , "").strip() for a, b in zip(data["Gender"].astype(str), data["Name"].astype(str))]
    data['Brand'] = data['Name'].apply(lambda x: x.split(' ')[-1])
    data["Name"] = [b.replace(a , "").strip() for a, b in zip(data["Brand"].astype(str), data["Name"].astype(str))]

    ## Handle missing values

    data.dropna(subset=["Description"], inplace=True)
    data.fillna("0", inplace=True)
    data[["Rating Count"]] = data[["Rating Count"]].replace(",", "", regex=True)
    data["Description"] = data["Description"].fillna("")
    data["Main Accords"] = data["Main Accords"].fillna("")
    
    ## Convert data types

    data[["Rating Count"]] = data[["Rating Count"]].astype(int)
    data[["Rating Value"]] = data[["Rating Value"]].astype(float)

    ## Standardize Gender

    gender_map = {"for women": "women", "for men": "men", "for women and men": "unisex"}
    data["Gender"] = data["Gender"].replace(gender_map)

    ## Clean text columns

    for col in ["Main Accords", "Perfumers"]:
        data[col] = data[col].apply(clean_text)

    data["Main Accords"] = data["Main Accords"].apply(clean_accords)

    ## Compute weighted rating

    C = data["Rating Value"].mean()
    m = data["Rating Count"].quantile(0.75)
    data["Weighted Rating"] = data.apply(lambda x: bayesian_avg(x["Rating Value"], x["Rating Count"], C, m), axis=1)

    return  data[['Name', 'Gender', 'Main Accords', 'Description', 'url', 'Weighted Rating', 'Brand']]

In [7]:
for col in ["Main Accords", "Perfumers"]:
    data[col] = data[col].apply(clean_text)

data["Description"] = data["Description"].fillna("")
data["Main Accords"] = data["Main Accords"].fillna("")

In [8]:
df_clean = clean_data(data)

C:\Users\atliu\AppData\Local\Temp\ipykernel_38344\1141876072.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data.fillna("0", inplace=True)


## Genre Map

In [9]:
gender_map = {"for women": "women", "for men": "men", "for women and men": "unisex"}
df_clean["Gender"] = df_clean["Gender"].replace(gender_map)

In [10]:
df_clean.head()

,Name,Gender,Main Accords,Description,url,Weighted Rating,Brand
0,9am,women,citrus musky woody aromatic warm spicy la...,9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...,3.692447,Afnan
1,9am Dive,unisex,fruity woody green warm spicy aromatic ci...,9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...,4.220751,Afnan
2,9am pour Femme,women,fruity musky amber citrus powdery sweet ...,9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...,3.779070,Afnan
3,9pm,men,vanilla amber warm spicy cinnamon sweet f...,9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...,4.487696,Afnan
4,9pm pour Femme,women,woody aromatic rose fruity powdery violet...,9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...,3.572643,Afnan


In [11]:
df_clean.to_pickle("data.pkl")

# Embedding

In [12]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

## Genre Vector

In [13]:
GENDER_VECTORS = {
    "male":   np.array([1.0, 0.0]),
    "female": np.array([0.0, 1.0]),
    "unisex": np.array([1/np.sqrt(2), 1/np.sqrt(2)]),
}

In [14]:
def gender_to_vec(g):
    g = str(g).lower()
    return GENDER_VECTORS.get(g, np.array([0, 0, 1], dtype=float))

def expand_gender_vec(gv, dim=384):
    # Convert 3-dim gender vector → same dimensionality as BERT embedding
    return np.pad(gv, (0, dim - len(gv)), 'constant')


In [15]:
gender_embs = np.vstack([expand_gender_vec(gender_to_vec(g)) for g in df_clean["Gender"]])

## Text Embeddings

In [16]:
def embed_text(df, col, model):
    texts = df[col].fillna("").tolist()
    embeddings = model.encode(texts, show_progress_bar=True)
    return embeddings

In [17]:
desc_embs = embed_text(df_clean, "Description", model)
accord_embs = embed_text(df_clean, "Main Accords", model)

Batches: 100%|██████████| 2191/2191 [01:21<00:00, 26.77it/s] 


## Brand Embedding

In [18]:
def create_brand_matrix(df):
    data = df.copy()

    brands = sorted(data["Brand"].unique())
    brand_to_idx = {b: i for i, b in enumerate(brands)}

    brand_matrix = np.zeros((len(data), len(brands)), dtype=np.float32)

    for i, brand in enumerate(data["Brand"]):
        brand_matrix[i, brand_to_idx[brand]] = 1
    
    return brand_matrix


def generate_brand_embedings(df, shape = 384):
    brand_matrix = create_brand_matrix(df)

    pca = PCA(n_components=shape)
    brand_embs = pca.fit_transform(brand_matrix)
    return brand_embs

In [19]:
brand_embs = generate_brand_embedings(df_clean, shape=desc_embs.shape[1])

# Saving Embeddings

In [20]:
np.save("desc_embs.npy", desc_embs)
np.save("accord_embs.npy", accord_embs)
np.save("gender_embss.npy", gender_embs)
np.save("brand_embs.npy", brand_embs)